# EfficientNetB0: frozen-backbone baseline

**Status: starter notebook; no real-data training has been run in this file.**

This experiment trains the new classifier for 10 epochs using the team's
existing trainer, AdamW, cross-entropy and fixed train/validation manifests.
The best checkpoint is chosen by **validation loss**, matching the shared trainer.
The test set stays isolated until the group finalizes its experiments.

Complete the smoke-check notebook first. Keep the kernel running during training.
Read `docs/efficientnetb0_start_here.md` for setup and commit milestones.

## 1. Load configuration and select the device

In [ ]:
from pathlib import Path
import sys
import os
import json
import hashlib
import platform
import subprocess
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import yaml
import torch
import torchvision
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

# Works when opened from either the repository root or notebooks/.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'src/data/dataloader.py').is_file()
     and (p / 'configs/base.yaml').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook inside your cloned waste-classification repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.dataloader import create_dataloaders
from src.models.efficientnetb0 import create_efficientnetb0
from src.utils.seed import set_seed

with (PROJECT_ROOT / 'configs/base.yaml').open() as handle:
    base_config = yaml.safe_load(handle)
with (PROJECT_ROOT / 'configs/efficientnetb0.yaml').open() as handle:
    config = yaml.safe_load(handle)

set_seed(config['experiment']['seed'])
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Project:', PROJECT_ROOT)
print('Device:', device)
print('PyTorch:', torch.__version__, '| Torchvision:', torchvision.__version__)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('ImageNet weights will download automatically if not already cached.')

## 2. Verify the shared dataset and create loaders

In [ ]:
# Get the SAME image folder used by your teammate. Do not recreate the splits.
# You may change this path if the dataset is stored outside the repository.
DATASET_ROOT = Path(os.environ.get(
    'WASTE_DATASET_ROOT',
    str(PROJECT_ROOT / base_config['data']['dataset_root']),
)).expanduser()
MANIFEST_DIR = PROJECT_ROOT / base_config['data']['manifest_dir']

frames = {split: pd.read_csv(MANIFEST_DIR / f'{split}.csv')
          for split in ['train', 'val', 'test']}
expected_classes = base_config['classes']
assert config['model']['num_classes'] == len(expected_classes)
assert config['model']['freeze_backbone'] is True, 'This notebook is the frozen baseline.'
assert config['model']['pretrained'] is True, 'The baseline uses pretrained ImageNet weights.'
assert base_config['data']['image_size'] == 224, 'Shared transforms currently use 224 pixels.'

seen_paths = set()
for split, frame in frames.items():
    assert {'relative_path', 'class_name', 'split'} <= set(frame.columns)
    assert len(frame) > 0 and not frame.isna().any().any()
    assert frame['relative_path'].is_unique, f'Duplicate paths within {split}'
    assert frame['split'].eq(split).all(), f'Incorrect split labels in {split}'
    assert set(frame['class_name']) == set(expected_classes)
    paths = set(frame['relative_path'])
    assert not seen_paths.intersection(paths), f'Overlapping paths found in {split}'
    seen_paths.update(paths)
    assert all(Path(row.relative_path).parts[0] == row.class_name
               for row in frame.itertuples()), f'Folder/label mismatch in {split}'

summary = pd.DataFrame({'split': list(frames),
                        'images': [len(frame) for frame in frames.values()]})
display(summary)
print('CSV structure, class names and path disjointness passed.')
print('This does not verify near-duplicate content or the correctness of every label.')

# Check presence of files; no test image is loaded or used for model selection.
missing = [relative_path for relative_path in sorted(seen_paths)
           if not (DATASET_ROOT / relative_path).is_file()]
if missing:
    raise FileNotFoundError(
        f'{len(missing)} manifest images are missing under {DATASET_ROOT}. '
        f'Examples: {missing[:5]}. Obtain the exact shared dataset; do not rename '
        'files or regenerate the CSV splits to hide this error.'
    )
print('All manifest image paths exist.')

loaders = create_dataloaders(
    dataset_root=DATASET_ROOT,
    manifest_dir=MANIFEST_DIR,
    batch_size=config['training']['batch_size'],
    num_workers=0,
    pin_memory=(device.type == 'cuda'),
)
class_to_idx = loaders['class_to_idx']
class_names = loaders['train_dataset'].classes
assert class_names == expected_classes
train_loader = loaders['train_loader']
val_loader = loaders['val_loader']
print('Shared class mapping:', class_to_idx)

## 3. Create an experiment folder and record the exact settings

Each run gets its own folder. No results are assumed before execution. The
shared transforms use bilinear evaluation resizing; this is the team's common
baseline rather than the torchvision weight preset's bicubic resize.

In [ ]:
run_id = config['experiment']['name'] + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = PROJECT_ROOT / 'results/efficientnetb0' / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
checkpoint_path = RUN_DIR / 'best.pth'

try:
    source_commit = subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True, stderr=subprocess.DEVNULL).strip()
    working_tree_dirty = bool(subprocess.check_output(
        ['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, text=True).strip())
except (OSError, subprocess.CalledProcessError):
    source_commit, working_tree_dirty = None, None

metadata = {
    'run_id': run_id,
    'training_complete': False,
    'started_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_commit': source_commit,
    'working_tree_dirty_at_start': working_tree_dirty,
    'python': platform.python_version(),
    'platform': platform.platform(),
    'torch': str(torch.__version__),
    'torchvision': str(torchvision.__version__),
    'device': str(device),
    'gpu_name': torch.cuda.get_device_name(0) if device.type == 'cuda' else None,
    'pretrained_weights': 'EfficientNet_B0_Weights.IMAGENET1K_V1',
    'class_to_idx': class_to_idx,
    'selection_metric': 'minimum validation loss',
    'frozen_backbone_batchnorm': 'eval; running statistics fixed',
    'preprocessing': 'shared src/data/transforms.py, 224 pixels, ImageNet normalization',
    'manifest_sha256': {
        name: hashlib.sha256((MANIFEST_DIR / f'{name}.csv').read_bytes()).hexdigest()
        for name in ['train', 'val', 'test']
    },
    'source_file_sha256': {
        name: hashlib.sha256((PROJECT_ROOT / name).read_bytes()).hexdigest()
        for name in ['src/data/transforms.py', 'src/training/trainer.py',
                     'src/models/efficientnetb0.py']
    },
}
(RUN_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n')
(RUN_DIR / 'config.yaml').write_text(yaml.safe_dump({'base': base_config, 'experiment': config}))
(RUN_DIR / 'class_to_idx.json').write_text(json.dumps(class_to_idx, indent=2) + '\n')
environment = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RUN_DIR / 'environment.txt').write_text(environment)
print('Run folder:', RUN_DIR.relative_to(PROJECT_ROOT))

## 4. Build the model, loss and optimizer

The feature extractor reuses ImageNet weights. Only the new classifier learns
in this baseline. Its dropout remains active during training. Do not add softmax
before `CrossEntropyLoss`.

In [ ]:
model = create_efficientnetb0(
    num_classes=len(class_names),
    pretrained=config['model']['pretrained'],
    freeze_backbone=config['model']['freeze_backbone'],
).to(device)
criterion = torch.nn.CrossEntropyLoss()
assert config['optimizer']['name'] == 'AdamW'
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=config['training']['learning_rate'],
    weight_decay=config['training']['weight_decay'],
)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 5. Train using the shared trainer

An epoch means one pass over all training images. This cell can take a while.
The trainer saves an improving checkpoint during the run. History is saved in
the next cell after all epochs finish; an interrupted run is not a completed
experiment. Use the epoch timings to estimate the remaining runtime.

In [ ]:
from src.training.trainer import train_model

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=config['training']['epochs'],
    checkpoint_path=checkpoint_path,
)

## 6. Save and plot the actual learning curves

In [ ]:
history_df = pd.DataFrame(history)
history_df.insert(0, 'epoch', np.arange(1, len(history_df) + 1))
history_df.to_csv(RUN_DIR / 'history.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for ax, metric, title in zip(axes, ['loss', 'accuracy'], ['Cross-entropy loss', 'Accuracy']):
    ax.plot(history_df['epoch'], history_df[f'train_{metric}'], label='Training', marker='o', color='#2459A9')
    ax.plot(history_df['epoch'], history_df[f'val_{metric}'], label='Validation', marker='s', color='#BC6A16')
    ax.set(xlabel='Epoch', ylabel=title, title=f'EfficientNetB0: {title}')
    ax.grid(alpha=0.2)
    ax.legend()
axes[1].set_ylim(0, 1)
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
fig.savefig(RUN_DIR / 'learning_curves.png', dpi=150)
plt.show()
display(history_df)

## 7. Restore the best checkpoint and evaluate validation performance

This evaluates the checkpoint chosen by validation loss, not necessarily the
last epoch. These are **validation** metrics; do not label them test results.

In [ ]:
from src.evaluation.metrics import collect_predictions, calculate_classification_metrics

# This checkpoint was created by this run. Do not load untrusted checkpoints.
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
y_true, y_pred, y_prob = collect_predictions(model, val_loader, device)
metrics, report, matrix = calculate_classification_metrics(y_true, y_pred, class_names)

(RUN_DIR / 'validation_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')
pd.DataFrame(report).T.to_csv(RUN_DIR / 'validation_report.csv')
pd.DataFrame(matrix, index=class_names, columns=class_names).to_csv(RUN_DIR / 'validation_confusion_matrix.csv')

predictions = frames['val'][['relative_path', 'class_name']].copy()
predictions['predicted_class'] = [class_names[int(i)] for i in y_pred]
predictions['top_class_score'] = y_prob.max(axis=1)
predictions.to_csv(RUN_DIR / 'validation_predictions.csv', index=False)

metadata.update({
    'training_complete': True,
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'best_epoch': int(checkpoint['epoch']),
    'best_val_loss': float(checkpoint['val_loss']),
    'total_epoch_time_seconds': float(history_df['epoch_time_seconds'].sum()),
    'validation_metrics': metrics,
})
(RUN_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n')
checkpoint['class_to_idx'] = class_to_idx
checkpoint['model_name'] = 'efficientnetb0'
torch.save(checkpoint, checkpoint_path)

print('Best epoch:', checkpoint['epoch'])
display(pd.DataFrame([metrics]))
display(pd.DataFrame(report).T.round(4))

## 8. Inspect which categories are confused

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(10, 9), constrained_layout=True)
ConfusionMatrixDisplay(matrix, display_labels=class_names).plot(
    ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False,
)
ax.set_title('EfficientNetB0: validation confusion matrix')
fig.savefig(RUN_DIR / 'validation_confusion_matrix.png', dpi=150)
plt.show()

mistakes = predictions[predictions['class_name'] != predictions['predicted_class']]
display(mistakes.head(10))
print('Saved results:', RUN_DIR.relative_to(PROJECT_ROOT))
print('Keep best.pth separately: the repository intentionally ignores model weights.')

## 9. Write your interpretation and commit the completed run

Replace these prompts with observations supported by your outputs:

- At which epoch was validation loss lowest?
- Did training improve while validation got worse? Explain possible overfitting.
- What are validation accuracy and macro F1? Why can they differ?
- Which two classes are most often confused? Inspect their validation images.
- What should the next controlled experiment change, and why?

Commit this saved notebook and this run's small CSV/JSON/PNG/config files.
Back up `best.pth` outside Git and record its location in your experiment notes.
Keep the test set untouched. Plan fine-tuning as the next separate experiment:
reload this best checkpoint, unfreeze selected/all features, create a new
optimizer with a smaller learning rate, and save to a new run directory.

**Comparison note:** this model holds frozen BatchNorm statistics fixed.
The current ResNet50 factory freezes parameter gradients only; ask the team
to align or explicitly document the difference when comparing frozen baselines.